# Add more features to the base cohort

In [ ]:

import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:

from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet
from  env.parameters import P
from util.logging import logging_setup

In [ ]:
import pandas as pd
import numpy as np
import  dask.dataframe as dd
pd.set_option('display.max_rows', 500)

In [ ]:
P.field_encoding_dict

# Load the core file and the base cohort


In [ ]:
# load the cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_1_base.csv''')
df_cohort.head()

In [ ]:
dd_all = dd.read_parquet(f'''{P.output_parquet_path}/{P.core_all_name}''')
cols_all = dd_all.columns


# Add ethnicity

In [ ]:
# Ethnic backgroud
search_term = "21000"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
df_eth = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_eth = df_eth.compute()
print(df_eth.head())
print(df_eth.dtypes)

In [ ]:
# typecase
# eid is non-nullable integer
# Ethnicity is nullable integer
df_eth['eid'] = df_eth['eid'].astype('int64')
df_eth[list_term] = df_eth[list_term].astype("Int64")
print(df_eth.dtypes)

In [ ]:
df_eth.head()

In [ ]:
df_eth[df_eth["21000-0.0"].isnull() & df_eth["21000-1.0"].notnull()]

In [ ]:
df_eth[df_eth["21000-3.0"].notnull()]


In [ ]:
df_eth[(df_eth["21000-0.0"].notnull()) & (df_eth["21000-1.0"].notnull()) & (df_eth["21000-0.0"] != df_eth["21000-1.0"])]


In [ ]:
df_eth_clean = df_eth.copy()
df_eth_clean['eth_code']= df_eth_clean["21000-3.0"].\
    fillna(df_eth_clean["21000-2.0"]).\
    fillna(df_eth_clean["21000-1.0"]).\
    fillna(df_eth_clean["21000-0.0"])

In [ ]:
df_eth_clean

In [ ]:
df_eth_clean[(df_eth_clean["21000-0.0"].notnull()) & (df_eth_clean["21000-1.0"].notnull()) & (df_eth_clean["21000-0.0"] != df_eth_clean["21000-1.0"])]


In [ ]:
df_eth_clean[df_eth_clean['eth_code']== -3]

In [ ]:
df_eth_clean[(df_eth_clean['eth_code']== -3) & (df_eth_clean['21000-0.0'] != -3)]


In [ ]:
df_eth_sel = df_eth_clean[['eid', 'eth_code']]
df_eth_sel

In [ ]:
# map
eth_coding = pd.read_csv(P.field_encoding_dict.get("eth").get("coding_file_path"), delimiter="\t")
eth_coding

In [ ]:
eth_map_dict = eth_coding.set_index("coding").to_dict()['meaning']
print(eth_map_dict)

In [ ]:
#df_eth_sel.loc[:, 'eth'] = df_eth_sel['eth_code'].map(eth_map_dict)
df_eth_sel = df_eth_sel.assign(eth=df_eth_sel['eth_code'].map(eth_map_dict))

df_eth_sel

In [ ]:
print(df_eth_sel.value_counts("eth", dropna=False))


In [ ]:
df_eth_sel[df_eth_sel['eth'].isnull()].shape

In [ ]:
df_eth_sel.loc[df_eth_sel['eth'].isnull(), 'eth'] = 'Unknown'
print(df_eth_sel.value_counts("eth", dropna=False))


In [ ]:
df_cohort_out = pd.merge(df_cohort, df_eth_sel, on = ["eid"], how= "left")
df_cohort_out

In [ ]:
# check nulls in eth_code
print(df_cohort_out.value_counts("eth_code", dropna=False))

In [ ]:
# check nulls in eth
print(df_cohort_out.value_counts("eth", dropna=False))

In [ ]:
def adjust_ethnicity_codes(df: pd.DataFrame, col_eth_code: str = "eth_code", col_eth: str = "eth") -> pd.DataFrame:
    
    # full -1, -3, and null with 0 for unknown. 
    df.loc[df[col_eth_code].isnull(), col_eth_code] = 0
    df.loc[df[col_eth_code]== -1, col_eth_code] = 0
    df.loc[df[col_eth]== "Do not know" , col_eth] = "Unknown"
    df.loc[df[col_eth_code]== -3, col_eth_code] = 0
    df.loc[df[col_eth]== "Prefer not to answer", col_eth] = "Unknown"
    return df

In [ ]:
df_cohort_out = adjust_ethnicity_codes(df_cohort_out, col_eth_code="eth_code", col_eth="eth")


In [ ]:
print(df_cohort_out.value_counts("eth", dropna=False))


In [ ]:
print(df_cohort_out.value_counts("eth_code", dropna=False))


# Add simpler ethnicities

In [ ]:
print(df_cohort_out.value_counts("eth", dropna=False))


In [ ]:
eth_dict_1 = {
    'British': 'White',
    'Any other white background': 'White',
    'White': 'White',
    'Irish': 'White',
    'Indian': 'Asian',
    'Other ethnic group': 'Other',
    'Caribbean': 'Other',
    'African': 'Black',
    'Unknown': 'Unknown',
    'Pakistani': 'Asian',
    'Any other Asian background': 'Asian',
    'Chinese': 'Asian',
    'Any other mixed background': 'Mixed',
    'White and Asian': 'Mixed',
    'White and Black Caribbean': 'Mixed',
    'White and Black African': 'Mixed',
    'Bangladeshi': 'Asian',
    'Asian or Asian British': 'Asian',
    'Black or Black British': 'Black',
    'Mixed': 'Mixed',
    'Any other Black background': 'Black'
}

eth_dict_2 = {
    'British': 'White',
    'Any other white background': 'White',
    'White': 'White',
    'Irish': 'White',
    'Indian': 'Non-white',
    'Other ethnic group': 'Non-white',
    'Caribbean': 'Non-white',
    'African': 'Non-white',
    'Unknown': 'Unknown',
    'Pakistani': 'Non-white',
    'Any other Asian background': 'Non-white',
    'Chinese': 'Non-white',
    'Any other mixed background': 'Non-white',
    'White and Asian': 'Non-white',
    'White and Black Caribbean': 'Non-white',
    'White and Black African': 'Non-white',
    'Bangladeshi': 'Non-white',
    'Asian or Asian British': 'Non-white',
    'Black or Black British': 'Non-white',
    'Mixed': 'Non-white',
    'Any other Black background': 'Non-white'
}

In [ ]:
df_cohort_out["eth_grouped_1"] =  df_cohort_out['eth'].map(eth_dict_1)
df_cohort_out["eth_grouped_2"] =  df_cohort_out['eth'].map(eth_dict_2)

In [ ]:
df_cohort_out.groupby(['eth', 'eth_grouped_1'], dropna=False).size().reset_index(name='Count')


In [ ]:
df_cohort_out.groupby(['eth', 'eth_grouped_2'], dropna=False).size().reset_index(name='Count')


# IMD

In [ ]:
# IMD England
search_term = "26410"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_imd_en = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_imd_en = dd_imd_en.compute()
print(df_imd_en.dtypes)
print(df_imd_en.head())

In [ ]:
# typecase
# eid is non-nullable integer
# imd is nullable float
df_imd_en['eid'] = df_imd_en['eid'].astype('int64')
df_imd_en[list_term] = df_imd_en[list_term].astype("float")
print(df_imd_en.head())
print(df_imd_en.dtypes)

In [ ]:
# IMD Scotland
search_term = "26427"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_imd_sct = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_imd_sct = dd_imd_sct.compute()
print(df_imd_sct.dtypes)
print(df_imd_sct.head())

In [ ]:
# typecase
# eid is non-nullable integer
# imd is nullable float
df_imd_sct['eid'] = df_imd_sct['eid'].astype('int64')
df_imd_sct[list_term] = df_imd_sct[list_term].astype("float")
print(df_imd_sct.dtypes)
print(df_imd_sct.head())

In [ ]:
# IMD Wales
search_term = "26426"
list_term = list_field_instances(col_list=cols_all, search_for_field=search_term)
print(list_term)
dd_imd_wales = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_imd_wales = dd_imd_wales.compute()
print(df_imd_wales.dtypes)
print(df_imd_wales.head())

In [ ]:
# typecase
# eid is non-nullable integer
# imd is nullable float
df_imd_wales['eid'] = df_imd_wales['eid'].astype('int64')
df_imd_wales[list_term] = df_imd_wales[list_term].astype("float")
print(df_imd_wales.dtypes)
print(df_imd_wales.head())

In [ ]:
# Rename and join
df_imd_en['imd_en'] = df_imd_en['26410-0.0']
df_imd_sct['imd_sct'] = df_imd_sct['26427-0.0']
df_imd_wales['imd_wales'] = df_imd_wales['26426-0.0']

df_imd_all = pd.merge(pd.merge(df_imd_en[["eid", "imd_en"]],
                               df_imd_sct[["eid", "imd_sct"]], on = ['eid'], how='outer'),
                      df_imd_wales[["eid", "imd_wales"]], on = ['eid'], how='outer')
df_imd_all




In [ ]:
# checks
print(df_imd_all[(df_imd_all['imd_en'].notnull()) & (df_imd_all['imd_sct'].notnull())].shape)
print(df_imd_all[(df_imd_all['imd_en'].notnull()) & (df_imd_all['imd_wales'].notnull())].shape)
print(df_imd_all[(df_imd_all['imd_sct'].notnull()) & (df_imd_all['imd_wales'].notnull())].shape)
print(df_imd_all[(df_imd_all['imd_sct'].isnull()) & (df_imd_all['imd_wales'].isnull()) & (df_imd_all['imd_en'].isnull())].shape)


In [ ]:
df_imd_all[(df_imd_all['imd_sct'].isnull()) & (df_imd_all['imd_wales'].isnull()) & (df_imd_all['imd_en'].isnull())]

In [ ]:
# add country
df_imd_all['country_imd']= 'Unknown'
df_imd_all.loc[df_imd_all['imd_en'].notnull(), 'country_imd'] = "England"
df_imd_all.loc[df_imd_all['imd_sct'].notnull(), 'country_imd'] = "Scotland"
df_imd_all.loc[df_imd_all['imd_wales'].notnull(), 'country_imd'] = "Wales"

df_imd_all['imd']= np.NAN
df_imd_all.loc[df_imd_all['imd_en'].notnull(), 'imd'] = df_imd_all['imd_en']
df_imd_all.loc[df_imd_all['imd_sct'].notnull(), 'imd'] = df_imd_all['imd_sct']
df_imd_all.loc[df_imd_all['imd_wales'].notnull(), 'imd'] = df_imd_all['imd_wales']

df_imd_all

In [ ]:
# merge
df_cohort_out = pd.merge(df_cohort_out, df_imd_all[["eid", "imd", "country_imd"]], on = ["eid"], how="left")

In [ ]:
# Quality control

In [ ]:
df_cohort_out.value_counts("country_imd", dropna=False)

In [ ]:
df_cohort_out

In [ ]:
print(df_cohort_out.value_counts("eth", dropna=False))

In [ ]:
print(df_cohort_out.value_counts("sex", dropna=False))


In [ ]:
# Null dob
print(df_cohort_out[df_cohort_out["dob"].isnull()].shape)


In [ ]:
# Death flag
print(df_cohort_out[(df_cohort_out['isin_deaths']==1) & (df_cohort_out["dod"].isnull())].shape)


In [ ]:
# Death flag
print(df_cohort_out[(df_cohort_out['isin_deaths']==0) & (df_cohort_out["dod"].notnull())].shape)


In [ ]:
# Birth after death
print(df_cohort_out[df_cohort_out['dob'] > df_cohort_out["dod"]].shape)


In [ ]:
# null IMD
print(df_cohort_out[df_cohort_out["imd"].isnull()].shape)


In [ ]:
df_cohort_out.to_csv(f'''{P.processed_data_path}/csv/cohort_2_advanced.csv''', index=False)


In [ ]:
df_cohort_out.shape

In [ ]:
# save to parquet
dd_cohort_out = dd.from_pandas(df_cohort_out, npartitions=2)

dask_to_parquet(dd_cohort_out, output_path=P.output_parquet_path, output_parquet_name="cohort_2_advanced", write_index=False)

In [ ]:
dd_test = dd.read_parquet(f'''{P.output_parquet_path}/cohort_2_advanced''')

In [ ]:
dd_test.head()

In [ ]:
dd_test[['eid']].shape[0].compute()

In [ ]:
dd_test.shape
